In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [68]:
import pandas as pd

df = pd.read_csv('../data/talabat_enhanced_orders.csv')
df.head()

,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,...,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,...,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,...,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,...,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online
3,4,U3612,420,32,Sushi,2,221.18,2025-06-13 02:30:00,2025-06-13 03:22:00,52,...,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
4,5,U3492,73,364,Koshary,5,355.55,2025-06-06 09:48:00,2025-06-06 10:32:00,44,...,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online


In [69]:
df.head()
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 23 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Order_ID                   100000 non-null  int64  
 1   User_ID                    100000 non-null  object 
 2   Restaurant_ID              100000 non-null  int64  
 3   Driver_ID                  100000 non-null  int64  
 4   Item_Name                  100000 non-null  object 
 5   Quantity                   100000 non-null  int64  
 6   Total_Price                100000 non-null  float64
 7   Order_Time                 100000 non-null  object 
 8   Delivery_Time              100000 non-null  object 
 9   Delivery_Duration_Minutes  100000 non-null  int64  
 10  City                       100000 non-null  object 
 11  Payment_Method             100000 non-null  object 
 12  Order_Status               100000 non-null  object 
 13  Driver_Vehicle             100

In [70]:
#1.check null values
 
df.isnull().sum()

Order_ID                     0
User_ID                      0
Restaurant_ID                0
Driver_ID                    0
Item_Name                    0
Quantity                     0
Total_Price                  0
Order_Time                   0
Delivery_Time                0
Delivery_Duration_Minutes    0
City                         0
Payment_Method               0
Order_Status                 0
Driver_Vehicle               0
Restaurant_Lat               0
Restaurant_Lon               0
Customer_Lat                 0
Customer_Lon                 0
Driver_Lat                   0
Driver_Lon                   0
Delivery_Distance_km         0
Traffic_Level                0
Driver_Availability          0
dtype: int64

In [71]:
#2.check duplicate values
 
df.duplicated().sum()

0

In [72]:
#3.convert date and time columns
 
df['Order_Time'] = pd.to_datetime(df['Order_Time'])
df['Delivery_Time'] = pd.to_datetime(df['Delivery_Time'])
 
df[['Order_Time', 'Delivery_Time']].head()

,Order_Time,Delivery_Time
0,2025-06-16 08:32:00,2025-06-16 09:11:00
1,2025-06-03 21:27:00,2025-06-03 22:00:00
2,2025-06-01 14:48:00,2025-06-01 15:26:00
3,2025-06-13 02:30:00,2025-06-13 03:22:00
4,2025-06-06 09:48:00,2025-06-06 10:32:00


In [73]:
#4.Creating time based Features.
df['order_hour'] = df['Order_Time'].dt.hour
df['order_day'] = df['Order_Time'].dt.day
df['order_month'] = df['Order_Time'].dt.month
df['order_weekday'] = df['Order_Time'].dt.weekday
df['is_weekend'] = df['order_weekday'].isin([5, 6]).astype(int)

In [74]:
#5.Creating delivery duration column
df['delivery_duration_mins'] = (df['Delivery_Time'] - df['Order_Time']).dt.total_seconds() / 60
 
df['delivery_duration_mins'].describe()

count    100000.000000
mean         37.520110
std          10.060876
min          15.000000
25%          30.000000
50%          38.000000
75%          45.000000
max          60.000000
Name: delivery_duration_mins, dtype: float64

In [75]:
#6.check categorical values
 
categorical_cols = ['Payment_Method', 'Traffic_Level', 'Order_Status', 'Driver_Vehicle']
for col in categorical_cols:
    if col in df.columns:
        print(col, df[col].value_counts())

Payment_Method Payment_Method
Cash           33528
Wallet         33250
Credit Card    33222
Name: count, dtype: int64
Traffic_Level Traffic_Level
Low       45826
High      29220
Medium    24954
Name: count, dtype: int64
Order_Status Order_Status
Delivered     85197
Cancelled      9812
In Transit     4991
Name: count, dtype: int64
Driver_Vehicle Driver_Vehicle
Bicycle      33393
Car          33362
Motorbike    33245
Name: count, dtype: int64


In [76]:
#7.location information
 
location_cols = ['Restaurant_Lat', 'Restaurant_Lon', 'Customer_Lat', 'Customer_Lon']
df[location_cols].describe()

,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,30.119015,31.063065,30.119060,31.062971
std,1.271638,0.487754,1.271674,0.487811
min,27.160900,29.898701,27.160900,29.898706
25%,30.023110,31.008774,30.023370,31.008460
50%,30.587306,31.209099,30.587054,31.208817
75%,31.026912,31.371676,31.027112,31.371856
max,31.220099,31.521997,31.220096,31.521995


In [77]:
#8.Removing unrealistic values
 
df = df[df['delivery_duration_mins'] >= 0]
 
#9. Filtering only Completed orders
 
df = df[df['Order_Status'] == 'Delivered']

In [78]:
# 9 Group data by Restaurant_ID to convert order-level data → node-level data

restaurant_demand = df.groupby('Restaurant_ID').agg({
    'Quantity': 'sum',                      # total demand per restaurant
    'delivery_duration_mins': 'mean',       # average delivery time
    'order_hour': 'mean'                    # average order time (hour)
}).reset_index()

# View the aggregated data
restaurant_demand.head()

,Restaurant_ID,Quantity,delivery_duration_mins,order_hour
0,1,301,37.755319,10.851064
1,2,247,36.823529,10.870588
2,3,201,36.611111,11.472222
3,4,280,38.471264,11.459770
4,5,238,37.500000,12.200000


In [79]:
# 10 Select features that will be used as input to GraphSAGE

node_features = restaurant_demand[['Quantity', 'delivery_duration_mins', 'order_hour']]

# Check features
node_features.head()

,Quantity,delivery_duration_mins,order_hour
0,301,37.755319,10.851064
1,247,36.823529,10.870588
2,201,36.611111,11.472222
3,280,38.471264,11.459770
4,238,37.500000,12.200000


In [80]:
# 11 Convert Restaurant_ID → numerical indices (required for graph models)

node_ids = restaurant_demand['Restaurant_ID'].unique()

# Create dictionary mapping: Restaurant_ID → index
node_mapping = {id: i for i, id in enumerate(node_ids)}

# Check mapping
list(node_mapping.items())[:5]

[(1, 0), (2, 1), (3, 2), (4, 3), (5, 4)]

In [81]:
# 12 Get average latitude and longitude for each restaurant

restaurant_locations = df.groupby('Restaurant_ID')[['Restaurant_Lat', 'Restaurant_Lon']].mean().reset_index()

# View locations
restaurant_locations.head()

,Restaurant_ID,Restaurant_Lat,Restaurant_Lon
0,1,29.988616,31.072152
1,2,30.126295,31.078475
2,3,30.150074,31.124903
3,4,30.221218,31.039706
4,5,30.291201,31.054879


In [82]:
# 13 Create edges based on distance

from sklearn.metrics.pairwise import euclidean_distances

# Extract coordinates
coords = restaurant_locations[['Restaurant_Lat', 'Restaurant_Lon']].values

# Compute distance matrix between all restaurants
dist_matrix = euclidean_distances(coords)

# Create edges list
edges = []

# Define threshold (distance for connecting nodes)
threshold = 0.01  # you can tune this later

# Connect nodes if distance is small
for i in range(len(coords)):
    for j in range(len(coords)):
        if i != j and dist_matrix[i][j] < threshold:
            edges.append([i, j])

# Check edges
len(edges)

3592

In [83]:
# Convert node features and edges into tensors

import torch

# Convert node features into tensor format
# These are the input features for each restaurant node
x = torch.tensor(node_features.values, dtype=torch.float)

# Convert edges into PyTorch Geometric format
# edge_index should have shape [2, number_of_edges]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

# Check tensor shapes
print("Node feature shape:", x.shape)
print("Edge index shape:", edge_index.shape)

Node feature shape: torch.Size([1000, 3])
Edge index shape: torch.Size([2, 3592])


In [84]:
# Create target variable
# We will predict total demand (Quantity) for each restaurant

y = torch.tensor(restaurant_demand['Quantity'].values, dtype=torch.float)

# Check target shape
print("Target shape:", y.shape)

Target shape: torch.Size([1000])


In [85]:
#Create training and testing indices

from sklearn.model_selection import train_test_split
import numpy as np

# Create indices for all nodes
indices = np.arange(len(restaurant_demand))

# Split node indices into training and testing sets
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# Convert indices into tensors
train_idx = torch.tensor(train_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

# Check sizes
print("Training nodes:", len(train_idx))
print("Testing nodes:", len(test_idx))

Training nodes: 800
Testing nodes: 200


In [86]:
# Check PyTorch Geometric first

import torch_geometric
print("PyTorch Geometric is installed")

PyTorch Geometric is installed


In [87]:
# Create graph data object

from torch_geometric.data import Data

# Create graph object
data = Data(x=x, edge_index=edge_index, y=y)

print(data)

Data(x=[1000, 3], edge_index=[2, 3592], y=[1000])


In [90]:
# Define GraphSAGE model

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGEModel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphSAGEModel, self).__init__()
        
        # First GraphSAGE layer
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        
        # Second GraphSAGE layer
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [91]:
# Initialize model

model = GraphSAGEModel(
    in_channels=x.shape[1],   # 3 features
    hidden_channels=16,
    out_channels=1            # regression output
)

print(model)

GraphSAGEModel(
  (conv1): SAGEConv(3, 16, aggr=mean)
  (conv2): SAGEConv(16, 1, aggr=mean)
)


In [92]:
# Optimizer + Loss

import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [127]:
# Train model

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    out = model(data.x, data.edge_index).squeeze()

    loss = criterion(out[train_idx], data.y[train_idx])

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.0015
Epoch 20, Loss: 3.9695
Epoch 40, Loss: 1.8889
Epoch 60, Loss: 0.2638
Epoch 80, Loss: 0.0331
Epoch 100, Loss: 0.0059
Epoch 120, Loss: 0.0016
Epoch 140, Loss: 0.0016
Epoch 160, Loss: 0.0015
Epoch 180, Loss: 0.0015


In [128]:
# Evaluate model

from sklearn.metrics import mean_absolute_error, mean_squared_error

model.eval()

with torch.no_grad():
    predictions = model(data.x, data.edge_index).squeeze()

y_true = data.y[test_idx].cpu().numpy()
y_pred = predictions[test_idx].cpu().numpy()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 0.024411926
RMSE: 0.03484594
